# 02 - Control cohort (no semaglutide, no bariatric surgery)

**Runs in:** Truveta Studio notebook environment only.

**What this notebook does**

Builds the *unexposed / control* delivery cohort using the same delivery,
gestational-age, weight and outcome definitions as notebook 01, but **without**
any exposure requirement. Notebook 05 labels this cohort `non_user`.

**Population snapshot:** `Delivery`

**Outputs** (written to `study.get_output_path(fs=True)`):

| File | Contents |
|---|---|
| `control_t1.csv` | main control cohort, one row per person (analytic file) |
| `control_df.csv` | intermediate cohort after weights + BMI, before outcome flags |
| `contrl_zcodecount.csv` | number of pregnancy Z-code encounters per person |
| `full_control_df.csv` | control cohort **without** the weight requirement (sensitivity) |

**Run order:** 01 -> 02 -> 03 -> 04 -> 05.

> The weight-window join here is written in **PySpark** rather than the pandas
> loop used in notebook 01, because the control cohort is orders of magnitude
> larger. The windows and the "closest measurement wins" rule are identical.

## 1. Setup and configuration

In [ ]:
from truveta.study import Client, OutputMode, display_df

import warnings
from datetime import datetime, timedelta
from typing import overload

import numpy as np
import pandas as pd
import pyspark.pandas as ps
import matplotlib.pyplot as plt
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import col, abs, datediff, row_number, date_sub, date_add, to_date

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================================
# CONFIG - edit here, not below
# ============================================================================

POPULATION_TITLE = "Delivery"

CODESETS = {
    "delivery":            "/definitions/delivery",
    "pregnancy_zcode":     "/definitions/pregnancy-zcode",
    "bmi":                 "https://library.truveta.com/o/truveta/d/tr-body-mass-index-bmi",
    "gestational_diabetes":"https://library.truveta.com/o/truveta-research/d/gestational-diabetes",
    "type2_diabetes":      "/definitions/type-2-diabetes",
    "gestational_htn":     "https://library.truveta.com/o/truveta-research/d/tr-gestational-hypertension",
    "hypertension":        "/definitions/hypertension",
    "preeclampsia":        "/definitions/preeclampsia",
    "csection":            "/definitions/c-section",
    "stillbirth":          "https://library.truveta.com/o/truveta/d/stillbirth-code-set",
    "excessive_fetal_wt":  "/definitions/excessive-fetal-weight",
    "prenatal":            "/definitions/prenatal",
    "hyperlipidemia":      "https://library.truveta.com/o/truveta-research/d/hyperlipidemia",
    "osa":                 "https://library.truveta.com/o/truveta-research/d/obstructive-sleep-apnea",
    "depression":          "https://library.truveta.com/o/truveta-research/d/major-depression",
}

ICD10 = {
    "intra_grow_restrict": ["O36.59", "Z36.4", "O36.5990", "O36.591",
                            "O36.592", "O36.593", "O36.599"],
    "primiparous":         ["Z34.00", "O09.611", "O09.511", "O09.512", "O09.513"],
    "multiparous":         ["O34.21", "O09.521", "O09.40", "O09.621",
                            "O09.41", "O09.42", "O09.43"],
    "prior_csection":      ["O34.212", "O34.211", "O34.21", "O34.219"],
    "prior_preterm_birth": ["Z87.51", "O09.21", "O09.211", "O09.212",
                            "O09.213", "O09.219"],
}

WEIGHT_LOINC = ["58229-6", "18833-4", "29463-7", "3141-9", "3142-7",
                "8341-0", "8349-3", "8350-1", "8351-9"]

# --- study windows (must match notebook 01) -------------------------------
DELIVERY_CUTOFF  = "2022-01-01"
WEIGHT_CUTOFF    = "2021-01-01"   # weights before this are not needed for these deliveries

GA_MIN_WEEKS     = 24
GA_MAX_WEEKS     = 42
PRETERM_WEEKS    = 37

ZCODE_MAX_LAG_DAYS = 300

PREPREG_WINDOW_WEEKS   = 12
PREDELIVERY_WINDOW_WKS = 4

LBS_LL, LBS_UL = 90, 700

# --- output file names ----------------------------------------------------
OUT_INTERMEDIATE  = "/control_df.csv"
OUT_COHORT        = "/control_t1.csv"
OUT_ZCODE_COUNT   = "/contrl_zcodecount.csv"
OUT_NO_WEIGHT     = "/full_control_df.csv"

In [ ]:
client = Client(output_mode=OutputMode.PandasOnSpark)
# client = Client(output_mode=OutputMode.PySpark)

study = client.get_study()
population = study.get_population(title=POPULATION_TITLE)
snapshot = population.get_latest_snapshot()

output_path_local = study.get_output_path(fs=True)

ps.set_option("compute.ops_on_diff_frames", True)

snapshot.get_tables()

## 2. Shared helper functions

Duplicated verbatim across the study notebooks so each one runs standalone.
Reference copy: `src/truveta_helpers.py`.

In [ ]:
@overload
def decode_concepts(df: pd.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> pd.DataFrame: ...
@overload
def decode_concepts(df: ps.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> ps.DataFrame: ...
@overload
def decode_concepts(df: DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> DataFrame: ...
def decode_concepts(df, drop_concepts: bool = True, columns: list[str] | None = None):
    """Replace every `*ConceptId` column with its human-readable concept name."""
    def should_decode(col: str, dtype: str) -> bool:
        if columns:
            return col in columns
        return col.endswith("ConceptId") and str(dtype) in ("int", "int32", "float64", "float32")

    column_names = set(df.columns)

    def target_col(col: str) -> str:
        name = col.removesuffix("ConceptId")
        if name == col and drop_concepts:
            return name
        while name in column_names:
            name = f"{name}Name"
        return name

    def safe_name(name: str) -> str:
        while name in column_names:
            name = f"{name}_tmp"
        return name

    final_order: list[str] = []

    if isinstance(df, pd.DataFrame):
        lookup = None
        for col, dtype in zip(df.columns, df.dtypes):
            if should_decode(col, str(dtype)):
                name = target_col(col)
                column_names.add(name)
                if lookup is None:
                    lookup = (spark.sql("SELECT ConceptId, ConceptName FROM Concept")
                              .toPandas().set_index("ConceptId").ConceptName)
                df[name] = df[col].map(lookup)
                if drop_concepts and name != col:
                    df = df.drop(columns=[col])
                else:
                    final_order.append(col)
                final_order.append(name)
            else:
                final_order.append(col)
        return df[final_order]

    return_pandas = False
    if isinstance(df, ps.DataFrame):
        return_pandas = True
        df = df.to_spark()

    concepts_s = spark.sql("SELECT ConceptId, ConceptName FROM Concept").cache()
    for col, dtype in df.dtypes:
        if should_decode(col, str(dtype)):
            name = target_col(col)
            column_names.add(name)
            if col == name and drop_concepts:
                tmp_name = safe_name(col)
                df = (df.withColumnRenamed(col, tmp_name)
                        .join(concepts_s.withColumnRenamed("ConceptId", tmp_name)
                                        .withColumnRenamed("ConceptName", name),
                              on=tmp_name, how="left")
                        .drop(tmp_name))
            else:
                df = df.join(concepts_s.withColumnRenamed("ConceptId", col)
                                       .withColumnRenamed("ConceptName", name),
                             on=col, how="left")
                if drop_concepts:
                    df = df.drop(col)
                else:
                    final_order.append(col)
            final_order.append(name)
        else:
            final_order.append(col)
    df = df.select(final_order)
    return df.pandas_api() if return_pandas else df


def match_code(df, codes_df):
    """Decode concept ids and keep only rows whose `Code` belongs to the code set."""
    code_name = decode_concepts(df)
    case_names = codes_df.ConceptName.to_pandas().tolist()
    return code_name[code_name["Code"].isin(case_names)]

In [ ]:
def load_condition_data(snapshot, codeset_url=None, code_set=None, codes="codes",
                        table_name="Condition", view_name="tbl_index_condition",
                        concept_map_table="ConditionCodeConceptMap",
                        concept_map_key="CodeConceptMapId", verbose=True):
    """Load one clinical table filtered to a code set and decode it to code names."""
    if code_set is None:
        if codeset_url is None:
            raise ValueError("Must provide either `codeset_url` or `code_set`")
        code_set = snapshot.codeset_from_prose(url=codeset_url, variable_name=codes)

    index_table = snapshot.load_filtered_table(table_name, code_set, view_name=view_name)
    if verbose:
        print(f"[{table_name}] Unique PersonId (after code filter):",
              index_table["PersonId"].nunique())

    df = ps.sql(f"""
        SELECT m.PersonId, m.RecordedDateTime, pm.*
        FROM {view_name} m
        JOIN {concept_map_table} pm ON m.{concept_map_key} = pm.Id
    """).to_pandas()

    matched_df = match_code(df, code_set)
    if verbose:
        print("Total matched rows:", len(matched_df))
        print("Unique PersonId (after match):", matched_df["PersonId"].nunique())

    return matched_df, matched_df["PersonId"].nunique()


def mark_condition_in_pregnancy(condition_df, delivery_df, col_name):
    """Flag people whose condition was recorded between `estimated_LMP` and delivery."""
    merged = condition_df.merge(
        delivery_df[["PersonId", "estimated_LMP", "delivery_date"]],
        on="PersonId", how="left")

    merged["in_pregnancy"] = (
        (merged["RecordedDateTime"] >= merged["estimated_LMP"]) &
        (merged["RecordedDateTime"] <= merged["delivery_date"])
    )

    flagged_ids = merged.loc[merged["in_pregnancy"], "PersonId"].drop_duplicates()
    condition_flag = pd.DataFrame({"PersonId": flagged_ids, col_name: True})

    delivery_df = delivery_df.merge(condition_flag, on="PersonId", how="left")
    delivery_df[col_name] = delivery_df[col_name].fillna(False)
    return delivery_df


def condition_before_pregnancy(condition_df, delivery_df, condition_col_name):
    """Flag people with a condition recorded strictly before `estimated_LMP`."""
    merged = condition_df.merge(
        delivery_df[["PersonId", "estimated_LMP"]], on="PersonId", how="left")

    merged["pre_pregnancy"] = merged["RecordedDateTime"] < merged["estimated_LMP"]

    before_preg = merged[merged["pre_pregnancy"]].drop_duplicates(subset="PersonId")
    delivery_df[condition_col_name] = delivery_df["PersonId"].isin(
        before_preg["PersonId"].unique())
    return delivery_df

## 3. Exclusion code sets: ectopic pregnancy and multiple gestation

In [ ]:
ectopics_code = snapshot.codeset_from_prose(url=CODESETS["delivery"],
                                            variable_name="ectopic_code")

ectopics_pro = snapshot.load_filtered_table("Procedure", ectopics_code,
                                            view_name="tbl_ectopics_pro")
print("ectopic procedures, unique persons:", ectopics_pro["PersonId"].nunique())

ectopics_con = snapshot.load_filtered_table("Condition", ectopics_code,
                                            view_name="tbl_ectopics_con")
print("ectopic conditions, unique persons:", ectopics_con["PersonId"].nunique())

df = ps.sql("""SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.*
               FROM tbl_ectopics_pro p
               JOIN ProcedureCodeConceptMap pm ON p.CodeConceptMapId = pm.Id""").to_pandas()
ectopics_pro = match_code(df, ectopics_code)

df = ps.sql("""SELECT p.PersonId, p.RecordedDateTime, pm.*
               FROM tbl_ectopics_con p
               JOIN ConditionCodeConceptMap pm ON p.CodeConceptMapId = pm.Id""").to_pandas()
ectopics_con = match_code(df, ectopics_code)

ectopics = pd.concat([ectopics_pro, ectopics_con])
print("ectopic (combined), unique persons:", ectopics.PersonId.nunique())

In [ ]:
multiple_code = snapshot.codeset_from_prose(url=CODESETS["delivery"],
                                            variable_name="multiple_code")
multiple_df = snapshot.load_filtered_table("Condition", multiple_code,
                                           view_name="tbl_multiple")
print("multiple gestation, unique persons:", multiple_df["PersonId"].nunique())

df = ps.sql("""SELECT p.PersonId, p.RecordedDateTime, pm.*
               FROM tbl_multiple p
               JOIN ConditionCodeConceptMap pm ON p.CodeConceptMapId = pm.Id""").to_pandas()
multiple_df = match_code(df, multiple_code)

## 4. Body weight measurements

Identical unit-harmonisation logic to notebook 01.

In [ ]:
concept = snapshot.load_sql_table("SELECT * FROM Concept",
                                  view_name="tbl_concept")[["ConceptId", "ConceptName"]]

weight_codes_s = snapshot.codeset("LOINC", "selfAndDescendants", *WEIGHT_LOINC)
study.create_view(weight_codes_s, view_name="tbl_weight_codes_s")

WEIGHT_COLS = ["PersonId", "EffectiveDateTime", "RecordedDateTime",
               "NormalizedValueConceptId", "NormalizedValueNumeric",
               "NormalizedValueUOMConceptId", "StatusConceptId", "EncounterId"]

lab_weights = snapshot.load_filtered_table("LabResult", weight_codes_s,
                                           view_name="tbl_lab_weights")[WEIGHT_COLS]
obs_weights = snapshot.load_filtered_table("Observation", weight_codes_s,
                                           view_name="tbl_obs_weights")[WEIGHT_COLS]

all_weights = ps.concat([obs_weights, lab_weights], ignore_index=True)
display_df(all_weights)

In [ ]:
for id_col, name_col in [("NormalizedValueConceptId", "NormalizedValueConcept"),
                         ("NormalizedValueUOMConceptId", "NormalizedValueUOMConcept"),
                         ("StatusConceptId", "StatusConcept")]:
    all_weights = (all_weights
                   .merge(concept, how="left", left_on=id_col, right_on="ConceptId")
                   .rename(columns={"ConceptName": name_col})
                   .drop(columns=["ConceptId"]))

EXCLUDED_UNITS = [
    "per liter", "per meter", "billion per liter", "centimeter", "degree",
    "foot (US)", "heart beats per minute", "inches", "liter", "liter per minute",
    "lumen", "meter", "millimeter mercury column", "millivolt", "minute",
    "per hour", "percent", "second", "week", "Each", "Inches",
    "inch (international)", "each",
]
all_weights = all_weights[~all_weights["NormalizedValueUOMConcept"].isin(EXCLUDED_UNITS)]

all_weights["NormalizedValueUOMConcept"] = all_weights["NormalizedValueUOMConcept"].replace({
    "No Information": "unknown",
    "Field has not been mapped": "unknown",
    "Field is not present in source": "unknown",
    "Invalid": "unknown",
})

all_weights.groupby("NormalizedValueUOMConcept").size().reset_index()

In [ ]:
POUND_UNITS = ["pound (US and British)", "pound (US)", "pound (apothecary)"]

all_weights["NormalizedValueNumeric"] = all_weights["NormalizedValueNumeric"].astype(float)
all_weights["UOM_assumed"] = np.nan

# 1. Known unit -> use it as-is.
all_weights.loc[all_weights["NormalizedValueUOMConcept"].isin(POUND_UNITS), "UOM_assumed"] = "pound"
all_weights.loc[
    (all_weights["NormalizedValueUOMConcept"] != "unknown") &
    (~all_weights["NormalizedValueUOMConcept"].isin(POUND_UNITS)),
    "UOM_assumed"] = all_weights["NormalizedValueUOMConcept"]

# 2. Unknown unit -> infer from magnitude.
all_weights.loc[
    (all_weights["NormalizedValueNumeric"] > LBS_LL * 453.6) &
    (all_weights["NormalizedValueNumeric"] <= LBS_UL * 453.6) &
    (all_weights["NormalizedValueUOMConcept"] == "unknown"), "UOM_assumed"] = "gram"

all_weights.loc[
    (all_weights["NormalizedValueNumeric"] > LBS_LL * 16) &
    (all_weights["NormalizedValueNumeric"] <= LBS_UL * 16) &
    (all_weights["NormalizedValueUOMConcept"] == "unknown"), "UOM_assumed"] = "ounce (avoirdupois)"

all_weights.loc[
    (all_weights["NormalizedValueNumeric"] < 125) &
    (all_weights["NormalizedValueUOMConcept"] == "unknown"), "UOM_assumed"] = "kilogram"

# 3. Convert to pounds, then kg.
all_weights["pounds"] = np.nan
all_weights.loc[all_weights["UOM_assumed"] == "pound", "pounds"] = all_weights["NormalizedValueNumeric"]
all_weights.loc[all_weights["UOM_assumed"] == "ounce (avoirdupois)", "pounds"] = all_weights["NormalizedValueNumeric"] / 16
all_weights.loc[all_weights["UOM_assumed"] == "kilogram", "pounds"] = all_weights["NormalizedValueNumeric"] * 2.205
all_weights.loc[all_weights["UOM_assumed"] == "gram", "pounds"] = all_weights["NormalizedValueNumeric"] / 453.6

# 4. Drop implausible values and derive kg.
all_weights.loc[(all_weights["pounds"] < LBS_LL) | (all_weights["pounds"] > LBS_UL), "pounds"] = np.nan
all_weights["kg"] = all_weights["pounds"] / 2.205

all_weights.head()

## 5. Delivery records and the index delivery

Delivery conditions and procedures are pooled. When the same person has both a
condition and a procedure record on the same date, the **procedure** record wins
(`priority` map below), because procedure codes carry the more specific delivery
mode. The index delivery is then the person's **first** qualifying delivery.

In [ ]:
delivery_concode = snapshot.codeset_from_prose(url=CODESETS["delivery"],
                                               variable_name="conditionCodes")
delivery_con = snapshot.load_filtered_table("Condition", delivery_concode,
                                            view_name="tbl_index_delivery_con")
print("delivery conditions, unique persons:", delivery_con["PersonId"].nunique())

df = ps.sql("""SELECT m.PersonId, m.RecordedDateTime, pm.*
               FROM tbl_index_delivery_con m
               JOIN ConditionCodeConceptMap pm ON m.CodeConceptMapId = pm.Id""").to_pandas()
delivery_con = match_code(df, delivery_concode)
print("rows / persons:", len(delivery_con), delivery_con.PersonId.nunique())

delivery_procode = snapshot.codeset_from_prose(url=CODESETS["delivery"],
                                               variable_name="procedureCodes")
delivery_pro = snapshot.load_filtered_table("Procedure", delivery_procode,
                                            view_name="tbl_index_delivery_pro")
print("delivery procedures, unique persons:", delivery_pro["PersonId"].nunique())

df = ps.sql("""SELECT m.PersonId, m.StartDateTime, pm.*
               FROM tbl_index_delivery_pro m
               JOIN ProcedureCodeConceptMap pm ON m.CodeConceptMapId = pm.Id""").to_pandas()
delivery_pro = match_code(df, delivery_procode)

delivery_con["source"] = "condition"
delivery_pro["source"] = "procedure"
delivery_pro.head()

In [ ]:
delivery_pro = delivery_pro.rename(columns={"StartDateTime": "RecordedDateTime"})

PRIORITY = {"procedure": 1, "condition": 2}   # lower wins on a tie

combined_df = pd.concat([delivery_con, delivery_pro], ignore_index=True)
combined_df["priority"] = combined_df["source"].map(PRIORITY)
combined_df["RecordedDateTime"] = pd.to_datetime(combined_df["RecordedDateTime"])

combined_df = (combined_df
               .sort_values(["PersonId", "RecordedDateTime", "priority"])
               .drop_duplicates(["PersonId", "RecordedDateTime"])
               .reset_index(drop=True))
combined_df["RecordedDateTime"] = combined_df["RecordedDateTime"].dt.date
print("delivery records / persons:", len(combined_df), combined_df.PersonId.nunique())

In [ ]:
print("delivery x ectopic  :", combined_df.merge(ectopics, on="PersonId", how="inner").PersonId.nunique())
print("delivery x multiple :", combined_df.merge(multiple_df, on="PersonId", how="inner").PersonId.nunique())

In [ ]:
combined_df_cleaned = combined_df[~combined_df["PersonId"].isin(ectopics["PersonId"])]
combined_df_cleaned = combined_df_cleaned[~combined_df_cleaned["PersonId"].isin(multiple_df["PersonId"])]

combined_df_cleaned = combined_df_cleaned.dropna()
combined_df_cleaned = combined_df_cleaned[
    combined_df_cleaned["RecordedDateTime"] >= datetime.strptime(DELIVERY_CUTOFF, "%Y-%m-%d").date()
].reset_index(drop=True)
print("after exclusions + cutoff:", len(combined_df_cleaned), combined_df_cleaned.PersonId.nunique())

# Index delivery = first qualifying delivery per person.
delivery_df = (combined_df_cleaned
               .sort_values(["PersonId", "RecordedDateTime"])
               .drop_duplicates(subset=["PersonId"], keep="first")
               [["PersonId", "RecordedDateTime", "Code"]]
               .reset_index(drop=True))
print("index deliveries:", delivery_df.shape, delivery_df.PersonId.nunique())
delivery_df.head()

## 6. Gestational age from pregnancy Z-codes

Identical derivation to notebook 01 section 7.

In [ ]:
zcodecode = snapshot.codeset_from_prose(url=CODESETS["pregnancy_zcode"],
                                        variable_name="zcodes")
zcode = snapshot.load_filtered_table("Condition", zcodecode, view_name="tbl_index_zcodes")
print("Z-code, unique persons:", zcode["PersonId"].nunique())

df = ps.sql("""SELECT m.PersonId, m.RecordedDateTime, pm.*
               FROM tbl_index_zcodes m
               JOIN ConditionCodeConceptMap pm ON m.CodeConceptMapId = pm.Id""").to_pandas()
zcode = match_code(df, zcodecode)
print("rows / persons:", len(zcode), zcode.PersonId.nunique())

In [ ]:
mask_remove = zcode["Code"].str.lower().isin([
    "weeks of gestation of pregnancy not specified",
    "less than 8 weeks gestation of pregnancy",
])
zcode = zcode.loc[~mask_remove].copy()

zcode["gestational_week_zcode"] = zcode["Code"].str.extract(r"(\d+)", expand=False).astype("float")

zcode = (zcode
         .sort_values(["PersonId", "RecordedDateTime", "gestational_week_zcode"])
         .groupby(["PersonId", "RecordedDateTime"], as_index=False)
         .tail(1))
zcode.gestational_week_zcode.describe()

In [ ]:
zcode = zcode.rename(columns={"RecordedDateTime": "zcodetime", "Code": "zcode"})

deliv_zcode = delivery_df.merge(zcode, on="PersonId", how="inner")
print("delivery x zcode rows / persons:", len(deliv_zcode), deliv_zcode.PersonId.nunique())

deliv_zcode["RecordedDateTime"] = pd.to_datetime(deliv_zcode["RecordedDateTime"], errors="coerce")
deliv_zcode["zcodetime"] = pd.to_datetime(deliv_zcode["zcodetime"], errors="coerce")

deliv_zcode = deliv_zcode[deliv_zcode["zcodetime"] <= deliv_zcode["RecordedDateTime"]].copy()
deliv_zcode["diff_days"] = (deliv_zcode["RecordedDateTime"] - deliv_zcode["zcodetime"]) / pd.Timedelta(days=1)
deliv_zcode = deliv_zcode[deliv_zcode["diff_days"] <= ZCODE_MAX_LAG_DAYS].copy()

deliv_zcode["zcodetime_date"] = deliv_zcode["zcodetime"].dt.date
deliv_zcode = (deliv_zcode
               .sort_values(["PersonId", "RecordedDateTime", "zcodetime_date", "gestational_week_zcode"])
               .groupby(["PersonId", "RecordedDateTime", "zcodetime_date"], as_index=False)
               .tail(1))

idx = deliv_zcode.groupby(["PersonId", "RecordedDateTime"])["diff_days"].idxmin()
closest_df = deliv_zcode.loc[idx].copy()
print("closest z-code per delivery:", len(closest_df), closest_df.PersonId.nunique())

zcode_counts = (deliv_zcode
                .groupby(["PersonId", "RecordedDateTime"])
                .size()
                .reset_index(name="zcode_count"))
print("zcode_counts:", zcode_counts.shape, zcode_counts.PersonId.nunique())

In [ ]:
delivery_df = closest_df.copy()

delivery_df["estimated_LMP"] = (
    delivery_df["zcodetime"] - pd.to_timedelta(delivery_df["gestational_week_zcode"] * 7, unit="D"))

delivery_df["gestational_week"] = (
    (delivery_df["RecordedDateTime"] - delivery_df["estimated_LMP"]) / pd.Timedelta(days=7))
delivery_df["gestational_age_days_at_delivery"] = (
    delivery_df["RecordedDateTime"] - delivery_df["estimated_LMP"]).dt.days

delivery_df = delivery_df[(delivery_df["gestational_week"] >= GA_MIN_WEEKS) &
                          (delivery_df["gestational_week"] <= GA_MAX_WEEKS)].copy()
print(delivery_df.gestational_week.describe())

delivery_df["preterm"] = (delivery_df["gestational_week"] < PRETERM_WEEKS).astype(int)
print(delivery_df["preterm"].value_counts())

delivery_df = delivery_df.merge(zcode_counts[["PersonId", "zcode_count"]],
                                on="PersonId", how="inner")

zcode_count_df = delivery_df[["PersonId", "zcode_count"]].copy()
zcode_count_df.to_csv(output_path_local + OUT_ZCODE_COUNT, index=False)

# Control cohort *before* the weight requirement - sensitivity sample (section 11).
full_control_df = delivery_df.copy()
print("control cohort before weight requirement:", full_control_df.shape)

In [ ]:
mode_week = delivery_df["gestational_week"].round().mode()[0]
plt.figure()
delivery_df["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Control cohort: gestational age at delivery")
plt.legend()
plt.show()

## 7. Attaching weights and computing gestational weight gain

PySpark implementation of the same two windows used in notebook 01:

* `prepreg_weight` - closest measurement within +/- `PREPREG_WINDOW_WEEKS` of the estimated LMP
* `predelivery_weight` - closest measurement in the `PREDELIVERY_WINDOW_WKS` weeks before delivery

There is no `pretreatment_weight` for controls: there is no treatment to anchor on.

In [ ]:
def add_weight_info(delivery_sdf, weights_sdf):
    """Attach the closest pre-pregnancy and pre-delivery weight to each delivery.

    Both arguments are Spark DataFrames. Returns a Spark DataFrame with
    `prepreg_weight`, `predelivery_weight` and the two `has_*` flags.
    """
    # Window bounds.
    delivery_sdf = (delivery_sdf
                    .withColumn("start_con", date_sub(col("estimated_LMP"), 7 * PREPREG_WINDOW_WEEKS))
                    .withColumn("end_con", date_add(col("estimated_LMP"), 7 * PREPREG_WINDOW_WEEKS))
                    .withColumn("start_del", date_sub(col("RecordedDateTime"), 7 * PREDELIVERY_WINDOW_WKS)))

    # --- closest weight around the estimated LMP --------------------------
    weights_con = weights_sdf.alias("w").join(
        delivery_sdf.alias("d"),
        (col("w.PersonId") == col("d.PersonId")) &
        (((col("w.RecordedDateTime") >= col("d.start_con")) & (col("w.RecordedDateTime") < col("d.estimated_LMP"))) |
         ((col("w.RecordedDateTime") > col("d.estimated_LMP")) & (col("w.RecordedDateTime") <= col("d.end_con")))),
        how="inner",
    ).withColumn("date_diff_con", abs(datediff(col("w.RecordedDateTime"), col("d.estimated_LMP"))))

    window_con = Window.partitionBy("d.PersonId", "d.estimated_LMP").orderBy(col("date_diff_con"))
    weights_con = weights_con.withColumn("rn_con", row_number().over(window_con)).filter(col("rn_con") == 1)

    # --- closest weight in the weeks before delivery ----------------------
    weights_del = weights_sdf.alias("w").join(
        delivery_sdf.alias("d"),
        (col("w.PersonId") == col("d.PersonId")) &
        (col("w.RecordedDateTime") >= col("d.start_del")) &
        (col("w.RecordedDateTime") <= col("d.RecordedDateTime")),
        how="inner",
    ).withColumn("date_diff_del", abs(datediff(col("w.RecordedDateTime"), col("d.RecordedDateTime"))))

    window_del = Window.partitionBy("d.PersonId", "d.RecordedDateTime").orderBy(col("date_diff_del"))
    weights_del = weights_del.withColumn("rn_del", row_number().over(window_del)).filter(col("rn_del") == 1)

    # --- attach both back onto the delivery frame -------------------------
    result = (delivery_sdf.alias("d")
              .join(weights_con.select(col("d.PersonId").alias("PersonId"),
                                       col("d.estimated_LMP").alias("estimated_LMP"),
                                       col("w.kg").alias("prepreg_weight")),
                    on=["PersonId", "estimated_LMP"], how="left")
              .join(weights_del.select(col("d.PersonId").alias("PersonId"),
                                       col("d.RecordedDateTime").alias("RecordedDateTime"),
                                       col("w.kg").alias("predelivery_weight")),
                    on=["PersonId", "RecordedDateTime"], how="left"))

    return (result
            .withColumn("has_prepreg_weight", col("prepreg_weight").isNotNull())
            .withColumn("has_predelivery_weight", col("predelivery_weight").isNotNull()))

In [ ]:
all_weights_df = all_weights[["PersonId", "RecordedDateTime", "pounds", "kg"]].copy()
all_weights_df["RecordedDateTime"] = all_weights_df["RecordedDateTime"].dt.date
all_weights_df = all_weights_df.dropna()

spark_weights_df = (all_weights_df.to_spark()
                    .withColumn("RecordedDateTime", to_date(col("RecordedDateTime")))
                    .filter(col("RecordedDateTime") >= WEIGHT_CUTOFF))
delivery_df_ps = spark.createDataFrame(delivery_df)

delivery_df_pd = add_weight_info(delivery_df_ps, spark_weights_df).toPandas()
print(delivery_df_pd["has_prepreg_weight"].value_counts())
print(delivery_df_pd["has_predelivery_weight"].value_counts())

In [ ]:
# Require both anchoring weights, then derive gestational weight gain.
delivery_df_pd["has_both_weights"] = (
    delivery_df_pd["has_prepreg_weight"] & delivery_df_pd["has_predelivery_weight"])
result_df = delivery_df_pd[delivery_df_pd["has_both_weights"]].copy()
result_df["gestation_weight"] = result_df["predelivery_weight"] - result_df["prepreg_weight"]
print("cohort with GWG:", result_df.shape)

In [ ]:
# Drop deliveries whose delivery code text says preterm/premature (same rule as
# notebook 01) - gestational age is taken from the Z-codes instead.
preterm_coded = result_df[result_df["Code"].str.contains("Premature|Preterm", case=False)]
print("preterm-coded deliveries dropped:", preterm_coded.PersonId.nunique())

result_df = result_df[~result_df["PersonId"].isin(preterm_coded["PersonId"])]
result_df = result_df.merge(zcode_counts[["PersonId", "zcode_count"]], on="PersonId", how="inner")
print("after dropping preterm-coded:", result_df.shape)

In [ ]:
KEEP_COLS = [
    "PersonId", "RecordedDateTime", "estimated_LMP",
    "gestational_week", "gestational_age_days_at_delivery", "preterm", "zcode_count",
    "start_del", "prepreg_weight", "predelivery_weight",
    "has_prepreg_weight", "has_predelivery_weight", "has_both_weights",
    "gestation_weight",
]
result_df = result_df[KEEP_COLS]
result_df.to_csv(output_path_local + OUT_INTERMEDIATE, header=True)
result_df.head()

## 8. BMI

In [ ]:
bmi_df = snapshot.codeset_from_prose(url=CODESETS["bmi"], variable_name="codes")
index_bmi_df = snapshot.load_filtered_table("Observation", bmi_df, view_name="tbl_index_bmi")
print("BMI observations, unique persons:", index_bmi_df["PersonId"].nunique())
index_bmi_df_lab = snapshot.load_filtered_table("LabResult", bmi_df, view_name="tbl_index_bmilab")

index_bmi_df = ps.concat([index_bmi_df, index_bmi_df_lab])
index_bmi_df = (index_bmi_df[["NormalizedValueNumeric", "PersonId", "RecordedDateTime"]]
                .sort_values(["PersonId", "RecordedDateTime"])
                .reset_index(drop=True)
                .dropna(subset=["NormalizedValueNumeric", "RecordedDateTime"])
                .drop_duplicates(subset=["PersonId", "RecordedDateTime"], keep="first")
                .to_pandas())
index_bmi_df["RecordedDateTime"] = ps.to_datetime(index_bmi_df["RecordedDateTime"])
print("persons with BMI:", index_bmi_df.PersonId.nunique())

In [ ]:
delivery_df_pd = pd.read_csv(output_path_local + OUT_INTERMEDIATE)
delivery_df_pd = delivery_df_pd.rename(columns={"RecordedDateTime": "delivery_date"})

index_bmi_df["NormalizedValueNumeric"] = index_bmi_df["NormalizedValueNumeric"].astype(float)
index_bmi_df = index_bmi_df.dropna(subset=["NormalizedValueNumeric"])
index_bmi_df = index_bmi_df[index_bmi_df["NormalizedValueNumeric"] > 10.0]

merged_df = pd.merge(index_bmi_df, delivery_df_pd, on="PersonId")
for c in ["RecordedDateTime", "estimated_LMP", "delivery_date"]:
    merged_df[c] = pd.to_datetime(merged_df[c])


def closest_bmi(df, anchor_col, out_name):
    """BMI measurement closest in time to `anchor_col`, one row per person."""
    tmp = df.copy()
    tmp["TimeDiff"] = (tmp[anchor_col] - tmp["RecordedDateTime"]).abs()
    return (tmp.sort_values(["PersonId", "TimeDiff"])
               .drop_duplicates("PersonId", keep="first")[["PersonId", "NormalizedValueNumeric"]]
               .rename(columns={"NormalizedValueNumeric": out_name}))


pre_bmi_df = closest_bmi(merged_df, "estimated_LMP", "PrePregnancyBMI")
post_bmi_df = closest_bmi(merged_df, "delivery_date", "nearDeliveryBMI")

delivery_df_pd = (delivery_df_pd
                  .merge(pre_bmi_df, on="PersonId", how="left")
                  .merge(post_bmi_df, on="PersonId", how="left"))
delivery_df_pd.to_csv(output_path_local + OUT_INTERMEDIATE, header=True)
delivery_df_pd.head()

## 9. Outcome and covariate code sets

Same definitions as notebook 01.

In [ ]:
gest_diabet, _ = load_condition_data(snapshot, codeset_url=CODESETS["gestational_diabetes"],
                                     view_name="tbl_index_gest_diabet")

type2_diabet, _ = load_condition_data(snapshot, codeset_url=CODESETS["type2_diabetes"],
                                      view_name="tbl_index_type2_diabet")

gest_hyper, _ = load_condition_data(snapshot, codeset_url=CODESETS["gestational_htn"],
                                    view_name="tbl_index_gest_hyper")

hypertension, _ = load_condition_data(snapshot, codeset_url=CODESETS["hypertension"],
                                      codes="conditionCodes", view_name="tbl_index_hyer")

preeclampsia, _ = load_condition_data(snapshot, codeset_url=CODESETS["preeclampsia"],
                                      view_name="tbl_index_preeclampsia")

stillbirth, _ = load_condition_data(snapshot, codeset_url=CODESETS["stillbirth"],
                                    view_name="tbl_index_sb")

excessive_fetal_weight, _ = load_condition_data(snapshot, codeset_url=CODESETS["excessive_fetal_wt"],
                                                view_name="tbl_index_efw")

Prenatal, _ = load_condition_data(snapshot, codeset_url=CODESETS["prenatal"],
                                  codes="Prenatal", view_name="tbl_index_Prenatal")

hyperlipidemia, _ = load_condition_data(snapshot, codeset_url=CODESETS["hyperlipidemia"],
                                        view_name="tbl_index_hyerlip")

osa, _ = load_condition_data(snapshot, codeset_url=CODESETS["osa"],
                             codes="osaCodes", view_name="tbl_index_osa")

depression, _ = load_condition_data(snapshot, codeset_url=CODESETS["depression"],
                                    view_name="tbl_index_mdep")

In [ ]:
csection_code = snapshot.codeset_from_prose(url=CODESETS["csection"], variable_name="codes")
index_c = snapshot.load_filtered_table("Procedure", csection_code, view_name="tbl_index_c")
print("c-section, unique persons:", index_c["PersonId"].nunique())

df = ps.sql("""SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.*
               FROM tbl_index_c p
               JOIN ProcedureCodeConceptMap pm ON p.CodeConceptMapId = pm.Id""").to_pandas()
csection = match_code(df, csection_code)
csection["StartDateTime"] = csection["StartDateTime"].fillna(csection["RecordedDateTime"])
csection = (csection.drop(columns=["RecordedDateTime"])
                    .rename(columns={"StartDateTime": "RecordedDateTime"}))
csection.head()

In [ ]:
intra_grow_restrict, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["intra_grow_restrict"]),
    view_name="tbl_index_intra_grow_restrict")

primiparous, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["primiparous"]),
    view_name="tbl_index_primiparous")

multiparous, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["multiparous"]),
    view_name="tbl_index_multiparous")

prior_Csection, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["prior_csection"]),
    view_name="tbl_index_prior_Csection")

Prior_Preterm_Birth, _ = load_condition_data(
    snapshot, code_set=snapshot.codeset("ICD10CM", "selfAndDescendants", *ICD10["prior_preterm_birth"]),
    view_name="tbl_index_Prior_Preterm_Birth")

## 10. Building the analytic variables

`delivery_df_t3` is the main control cohort (weight requirement applied).
`full_control_df` is the no-weight-requirement sensitivity cohort.

In [ ]:
delivery_df_pd = pd.read_csv(output_path_local + OUT_INTERMEDIATE)
full_control_df = full_control_df.rename(columns={"RecordedDateTime": "delivery_date"})

IN_PREGNANCY_FLAGS = [
    (gest_diabet, "gest_diabet"),
    (gest_hyper, "gest_hyper"),
    (preeclampsia, "preeclampsia"),
    (csection, "csection"),
    (excessive_fetal_weight, "excessive_fetal_weight"),
    (intra_grow_restrict, "intra_grow_restrict"),
]

delivery_df_t3 = delivery_df_pd
for condition_df, col_name in IN_PREGNANCY_FLAGS:
    delivery_df_t3 = mark_condition_in_pregnancy(condition_df, delivery_df_t3, col_name)
    full_control_df = mark_condition_in_pregnancy(condition_df, full_control_df, col_name)

delivery_df_t3.head()

In [ ]:
BEFORE_PREGNANCY_FLAGS = [
    (type2_diabet, "t2d_before_pregnancy"),
    (hypertension, "hyper_before_pregnancy"),
    (hyperlipidemia, "hyperlipid"),
    (depression, "depression"),
    (osa, "osa"),
]

for condition_df, col_name in BEFORE_PREGNANCY_FLAGS:
    delivery_df_t3 = condition_before_pregnancy(condition_df, delivery_df_t3, col_name)
    full_control_df = condition_before_pregnancy(condition_df, full_control_df, col_name)

print(delivery_df_t3.t2d_before_pregnancy.value_counts())
print(delivery_df_t3.hyper_before_pregnancy.value_counts())

In [ ]:
for frame in (delivery_df_t3, full_control_df):
    # obstetric history (any time)
    frame["prior_Csection"] = frame["PersonId"].isin(prior_Csection["PersonId"])
    frame["Prior_Preterm_Birth"] = frame["PersonId"].isin(Prior_Preterm_Birth["PersonId"])

    # parity
    frame["parity"] = "Unknown"
    frame.loc[frame["PersonId"].isin(primiparous["PersonId"]), "parity"] = "Primiparous"
    frame.loc[frame["PersonId"].isin(multiparous["PersonId"]), "parity"] = "Multiparous"

    # derived delivery / infant labels
    frame["delivery_type"] = np.where(frame["csection"] == True, "C-section", "Vaginal")
    frame["infant_gest_age_class"] = np.where(
        frame["excessive_fetal_weight"] == True, "Excessive", "Average")

    # incident (pregnancy-onset) conditions
    frame["gest_diabetes_no_prior_t2d"] = frame["gest_diabet"] & ~frame["t2d_before_pregnancy"]
    frame["gest_hyper_no_prior_hyper"] = frame["gest_hyper"] & ~frame["hyper_before_pregnancy"]
    frame["preeclampsia_no_prior_hyper"] = frame["preeclampsia"] & ~frame["hyper_before_pregnancy"]

print(delivery_df_t3.delivery_type.value_counts())
print(delivery_df_t3.gest_diabetes_no_prior_t2d.value_counts())

> **Known definition difference.** For the control cohort `obstetric_care` is
> "has any prenatal-care record at any time", whereas notebook 01 defines it for
> the exposed cohort as "has a prenatal-care record **during** the pregnancy
> window". Keep this in mind when interpreting the variable across arms; the
> column is also written under the legacy name `Obstetriccare` for backwards
> compatibility with earlier exports.

In [ ]:
for name in ["delivery_df_t3", "full_control_df"]:
    frame = globals()[name]
    frame["obstetric_care"] = frame["PersonId"].isin(Prenatal["PersonId"])
    frame["Obstetriccare"] = frame["obstetric_care"]   # legacy column name
    globals()[name] = frame

delivery_df_t3.obstetric_care.value_counts()

In [ ]:
# --- exclude stillbirths --------------------------------------------------
delivery_df_t3 = delivery_df_t3[~delivery_df_t3["PersonId"].isin(stillbirth["PersonId"])]
full_control_df = full_control_df[~full_control_df["PersonId"].isin(stillbirth["PersonId"])]
print("after stillbirth exclusion:",
      delivery_df_t3["PersonId"].nunique(), full_control_df["PersonId"].nunique())

## 11. Demographics: age, race/ethnicity, income

In [ ]:
df = ps.sql("SELECT * FROM Person").to_pandas()
person = decode_concepts(df).rename(columns={"Id": "PersonId"})

df = ps.sql("SELECT * FROM PersonRace").to_pandas()
race = decode_concepts(df)

delivery_df_t3 = (delivery_df_t3
                  .merge(person[["PersonId", "BirthDateTime", "Ethnicity", "Gender"]], on="PersonId", how="left")
                  .merge(race[["PersonId", "Race"]], on="PersonId", how="left"))
full_control_df = (full_control_df
                   .merge(person[["PersonId", "BirthDateTime", "Ethnicity", "Gender"]], on="PersonId", how="left")
                   .merge(race[["PersonId", "Race"]], on="PersonId", how="left"))

In [ ]:
def calculate_age(event_date, dob):
    """Age in years between a date of birth and an event date."""
    return (event_date - dob).days / 365.25


for frame in (delivery_df_t3, full_control_df):
    for c in ["delivery_date", "BirthDateTime"]:
        frame[c] = pd.to_datetime(frame[c]).dt.date
    frame["age_at_delivery"] = frame.apply(
        lambda row: calculate_age(row["delivery_date"], row["BirthDateTime"]), axis=1)

print("median age at delivery:", delivery_df_t3.age_at_delivery.median())

In [ ]:
def combine_race_ethnicity(row):
    """Collapse Truveta `Race` + `Ethnicity` into a single analysis variable."""
    race = str(row["Race"]).strip().lower()
    ethnicity = str(row["Ethnicity"]).strip().lower()

    if pd.isna(race) or pd.isna(ethnicity):
        return "Unknown"
    if ethnicity == "hispanic or latino":
        return "Hispanic"
    if ethnicity == "not hispanic or latino":
        if race == "white":
            return "Non-Hispanic White"
        if race == "black or african american":
            return "Non-Hispanic Black"
        if race in ("asian", "american indian or alaska native",
                    "native hawaiian or other pacific islander", "other race"):
            return "Other"
        return "Unknown"
    return "Unknown"


delivery_df_t3["race_ethnicity"] = delivery_df_t3.apply(combine_race_ethnicity, axis=1)
full_control_df["race_ethnicity"] = full_control_df.apply(combine_race_ethnicity, axis=1)
delivery_df_t3.race_ethnicity.value_counts()

In [ ]:
# --- income from Social Determinants of Health ---------------------------
snapshot.load_sql_table("SELECT * FROM SocialDeterminantsOfHealth",
                        cache=True, view_name="tbl_sdoh")
snapshot.load_sql_table("SELECT ConceptId, ConceptName FROM tbl_concept",
                        cache=True, view_name="tbl_concept_sel")

snapshot.load_sql_table("""
    SELECT PersonId, EffectiveStartDateTime, AttributeConceptId,
           NormalizedValueNumeric, NormalizedValueConceptId
    FROM tbl_sdoh
""", cache=True, view_name="tbl_sdoh_2")

snapshot.load_sql_table("""
    SELECT s.PersonId, s.EffectiveStartDateTime, s.NormalizedValueNumeric,
           s.NormalizedValueConceptId, c.ConceptId, c.ConceptName AS Attribute
    FROM tbl_sdoh_2 s
    LEFT JOIN tbl_concept_sel c ON s.AttributeConceptId = c.ConceptId
""", cache=True, view_name="tbl_sdoh_3")

snapshot.load_sql_table("""
    SELECT s.PersonId, s.EffectiveStartDateTime, s.Attribute,
           c.ConceptName AS Attribute_Value_Categorical,
           s.NormalizedValueNumeric AS Attribute_Value_Numeric
    FROM tbl_sdoh_3 s
    LEFT JOIN tbl_concept_sel c ON s.NormalizedValueConceptId = c.ConceptId
""", cache=True, view_name="tbl_sdoh_4")

SDOH_5 = snapshot.load_sql_table("""
    WITH SortedData AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY PersonId, Attribute
                                  ORDER BY EffectiveStartDateTime DESC, Attribute) AS rn
        FROM tbl_sdoh_4
    )
    SELECT * FROM SortedData WHERE rn = 1
    ORDER BY PersonId, EffectiveStartDateTime, Attribute
""", cache=True, view_name="tbl_sdoh_5")

SDOH_wide = (SDOH_5.to_pandas()
             .pivot(index="PersonId", columns="Attribute",
                    values=["EffectiveStartDateTime", "Attribute_Value_Categorical",
                            "Attribute_Value_Numeric"])
             .reset_index())
SDOH_wide.columns = SDOH_wide.columns.map(lambda index: f"{index[0]}_{index[1]}")
SDOH_wide.rename({"PersonId_": "PersonId"}, axis=1, inplace=True)

person_SDOH = person.merge(SDOH_wide, how="left", on="PersonId")
person_SDOH.Attribute_Value_Categorical_EstimatedAnnualIncome.value_counts()

In [ ]:
INCOME_COL = "Attribute_Value_Categorical_EstimatedAnnualIncome"


def reclassify_income(bracket):
    """Collapse the SDOH annual-income bracket string into three analysis bands."""
    try:
        low = int(str(bracket).split("-")[0].replace(",", "").strip())
    except (ValueError, AttributeError):
        return "Unknown"
    if low <= 50000:
        return "≤50000"
    if low <= 80000:
        return "50001-80000"
    return ">80000"


for name in ["delivery_df_t3", "full_control_df"]:
    frame = globals()[name]
    frame = frame.merge(person_SDOH[["PersonId", INCOME_COL]], on="PersonId", how="left")
    frame[INCOME_COL] = frame[INCOME_COL].fillna("Unknown")
    frame["Income"] = frame[INCOME_COL].apply(reclassify_income)
    globals()[name] = frame

delivery_df_t3.Income.value_counts()

## 12. Export

In [ ]:
delivery_df_t3.to_csv(output_path_local + OUT_COHORT, index=False)
full_control_df.to_csv(output_path_local + OUT_NO_WEIGHT, index=False)

print("control cohort persons:", delivery_df_t3["PersonId"].nunique())
print("wrote:")
for f in (OUT_COHORT, OUT_INTERMEDIATE, OUT_ZCODE_COUNT, OUT_NO_WEIGHT):
    print("  ", output_path_local + f)

In [ ]:
print(delivery_df_t3.gestational_week.describe())
print(delivery_df_t3.preterm.value_counts())
print(delivery_df_t3.zcode_count.describe())

mode_week = delivery_df_t3["gestational_week"].round().mode()[0]
median_week = delivery_df_t3["gestational_week"].round(2).median()
plt.figure()
delivery_df_t3["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.axvline(median_week, linestyle="--", label=f"Median week: {median_week}")
plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Control cohort: gestational age at delivery")
plt.legend()
plt.show()

## 13. Descriptive Table 1 (control cohort)

In [ ]:
!pip install tableone
from tableone import TableOne

In [ ]:
table1 = delivery_df_t3.dropna(subset=["PrePregnancyBMI"]).copy()

T1_COLUMNS = [
    "preterm", "prepreg_weight", "predelivery_weight", "gestation_weight",
    "PrePregnancyBMI", "nearDeliveryBMI", "gest_diabet", "gest_hyper",
    "preeclampsia", "csection", "excessive_fetal_weight", "intra_grow_restrict",
    "delivery_type", "infant_gest_age_class", "parity", "t2d_before_pregnancy",
    "hyper_before_pregnancy", "osa", "hyperlipid", "depression",
    "gest_diabetes_no_prior_t2d", "preeclampsia_no_prior_hyper",
    "gest_hyper_no_prior_hyper", "obstetric_care", "age_at_delivery",
    "race_ethnicity", "Income",
]

T1_CATEGORICAL = [
    "preterm", "gest_diabet", "gest_hyper", "preeclampsia", "csection",
    "excessive_fetal_weight", "intra_grow_restrict", "delivery_type",
    "infant_gest_age_class", "parity", "t2d_before_pregnancy",
    "hyper_before_pregnancy", "osa", "hyperlipid", "depression",
    "gest_diabetes_no_prior_t2d", "preeclampsia_no_prior_hyper",
    "gest_hyper_no_prior_hyper", "obstetric_care", "race_ethnicity", "Income",
]

TableOne(table1, columns=T1_COLUMNS, categorical=T1_CATEGORICAL, pval=False)

---

## Appendix - superseded approaches (do not run)

### A1. Preterm birth from a dedicated code set

Superseded by the Z-code gestational-age definition
(`preterm = gestational_week < 37`), section 6.

In [ ]:
# --- SUPERSEDED - DO NOT RUN ---
# preterm_concode = snapshot.codeset_from_prose(url="/definitions/preterm-birth", variable_name="codes")
# preterm_con, _ = load_condition_data(snapshot, code_set=preterm_concode,
#                                      view_name="tbl_index_preterm")

### A2. Conception date estimated from a fixed gestational length

Superseded by the Z-code derived `estimated_LMP`.

In [ ]:
# --- SUPERSEDED - DO NOT RUN ---
# def estimate_conception_date(row):
#     weeks = 35 if row["is_preterm"] else 39
#     return row["RecordedDateTime"] - pd.Timedelta(weeks=weeks)
#
# delivery_df["estimated_LMP"] = delivery_df.apply(estimate_conception_date, axis=1)

### A3. Unmatched three-group regressions (control vs med vs surgery)

This was the first-pass outcome analysis. It is **superseded by notebook 05**,
which uses propensity-score-matched pairwise comparisons and the
continued/former/non-user exposure grouping. Kept for provenance.

Note it reads `test_t3.csv` (in the original notebook this path referred to an
earlier export name, `delivery_df_t3.csv`).

In [ ]:
# --- SUPERSEDED - see notebook 05 - DO NOT RUN ---
# test_df = pd.read_csv(output_path_local + "/test_t3.csv")
# drug_source_label = pd.read_csv(output_path_local + "/drug_source_label.csv")
#
# # Split med vs surgery and relabel the "no prior T2D" medication users.
# med_full_df = test_df[test_df.source_type == "med"].copy()
# med_noexppreg = med_full_df.drop("source_type", axis=1).merge(
#     drug_source_label, on="PersonId", how="right")
#
# # People who initiated medication during pregnancy are dropped.
# start_drug_preg = med_full_df[~med_full_df.PersonId.isin(med_noexppreg.PersonId)]
# test_df = test_df[~test_df.PersonId.isin(start_drug_preg.PersonId)].copy()
#
# control_df = pd.read_csv(output_path_local + "/control_t1.csv")
# control_df = control_df.dropna(subset=["PrePregnancyBMI"]).copy()
# control_df["source_type"] = "control"
#
# combined_df = pd.concat([test_df, control_df], ignore_index=True).rename(
#     columns={"source_type": "group"})
#
# binary_outcomes = ["is_preterm", "preeclampsia_no_prior_hyper", "csection",
#                    "excessive_fetal_weight", "intra_grow_restrict",
#                    "gest_diabetes_no_prior_t2d", "gest_hyper_no_prior_hyper"]
# continuous_outcomes = ["prepreg_weight", "predelivery_weight", "gestation_weight"]
#
# for c in binary_outcomes:
#     combined_df[c] = combined_df[c].astype(int)
#
# for c, ref_order in {
#     "group": ["control", "med", "med_wo_t2d", "surgery"],
#     "race_ethnicity": ["Non-Hispanic White", "Non-Hispanic Black", "Hispanic", "Other", "Unknown"],
#     "Income": ["≤50000", "50001-80000", ">80000", "Unknown"],
#     "parity": ["Primiparous", "Multiparous", "Unknown"],
# }.items():
#     combined_df[c] = pd.Categorical(combined_df[c], categories=ref_order, ordered=True)
#
# # run_models(...) / extract_and_save_model_results(...) then fit one logit or OLS
# # per outcome with covariates:
# #   C(group) + age_at_delivery + C(race_ethnicity) + C(Income) + PrePregnancyBMI
# #   + C(parity) + t2d_before_pregnancy + hyper_before_pregnancy + depression
# # dropping t2d_before_pregnancy for gestational diabetes and
# # hyper_before_pregnancy for the hypertensive outcomes.